In [3]:
# %% [markdown]
# # 텔롭 분류 테스트 (Qwen3.5-27B + SGLang)
# 셀 순서대로 실행하세요.
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

# %% 셀 1: 모델 다운로드 (최초 1회만)
from huggingface_hub import snapshot_download

snapshot_download("Qwen/Qwen3.5-9B")
print("✅ 모델 다운로드 완료!")

Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 427.12it/s]

✅ 모델 다운로드 완료!


In [1]:
# %% 셀 2: SGLang 서버 시작 (커널 살아있는 동안 유지)
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

import subprocess
import time
import requests
 
sglang_log = open("sglang_server.log", "w")

server_process = subprocess.Popen(
    [
        "python", "-m", "sglang.launch_server",
        "--model-path", "Qwen/Qwen3.5-9B",
        "--port", "8000",
        "--mem-fraction-static", "0.8",
        "--context-length", "16384",
        "--reasoning-parser", "qwen3",
        "--attention-backend", "triton",
    ],
    stdout=sglang_log,
    stderr=subprocess.STDOUT,
)
 
# 서버 뜰 때까지 대기
print("⏳ SGLang 서버 시작 중...")
for i in range(300):  # 최대 5분 대기
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print(f"✅ 서버 준비 완료! ({i}초)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("❌ 서버 시작 실패. 로그 확인:")
    server_process.terminate()
    print(server_process.stdout.read().decode()[-2000:])


⏳ SGLang 서버 시작 중...
✅ 서버 준비 완료! (30초)


In [5]:
# %% 셀 5: 전체 채널/영상 처리 (채널 단위 저장, 처리 완료 skip)
import os
import json
import random
import glob
import base64
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
import httpx

random.seed(42)

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(30.0, connect=5.0)
)

MODEL_NAME = "Qwen/Qwen3.5-9B"
OCR_DIR = "./data/3_ocr_results"
FRAME_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/4_is_telop_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_PROMPT = """You are a video frame analyzer that classifies OCR-detected text regions as either "telop" or "scene_text".

Telop: Text overlaid during post-production. Subtitles, captions, name plates, speech bubbles, decorative titles, reaction text, timestamps, channel logos, watermarks, and any graphical text composited onto the video frame. May be static or animated.

Scene text: Text physically existing in the 3D scene. Signage, clothing logos, book covers, product labels, screens within the scene, license plates, posters, handwritten notes, etc.

Key visual cues for telop:
- No perspective distortion (faces viewer regardless of camera angle)
- Consistent lighting unaffected by scene illumination
- Often has outline, drop shadow, stroke, or semi-transparent background
- Clean uniform font rendering
- May overlap scene objects without being occluded
- Designed for readability over video content

Key visual cues for scene text:
- Perspective distortion matching its surface
- Affected by scene lighting, shadows, reflections
- Surface texture visible beneath text
- May be partially occluded by other objects"""

OCR_SCORE_THRESHOLD = 0.4
TIMEOUT = 30
MAX_WORKERS = 100

# 통계
total_channels = 0
skipped_channels = 0
total_videos = 0
total_frames = 0
total_telop = 0
total_scene = 0
total_unknown = 0
total_regions = 0
errors = 0
times = []


def call_vlm(task):
    """단일 VLM 요청 처리 (스레드에서 실행)"""
    video_name = task["video_name"]
    img_b64 = task["img_b64"]
    user_text = task["user_text"]
    ocr_texts = task["ocr_texts"]

    t0 = time.time()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
                        {"type": "text", "text": user_text}
                    ]
                }
            ],
            max_tokens=2048,
            temperature=0.1,
            timeout=TIMEOUT,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        elapsed = time.time() - t0

        raw = response.choices[0].message.content.strip()
        cleaned = raw.removeprefix("```json").removesuffix("```").strip()
        classifications = json.loads(cleaned)

        cls_map = {item["id"]: item.get("classification", "unknown") for item in classifications}
        is_telop = [cls_map.get(i, "unknown") == "telop" for i in range(len(ocr_texts))]

        return {"success": True, "elapsed": elapsed, "is_telop": is_telop, "cls_map": cls_map, "task": task}

    except Exception as e:
        elapsed = time.time() - t0
        return {"success": False, "elapsed": elapsed, "error": str(e)[:80], "task": task}


for channel_entry in sorted(os.scandir(OCR_DIR), key=lambda e: e.name):
    if not channel_entry.is_dir():
        continue
    channel_name = channel_entry.name
    total_channels += 1

    # 이미 처리된 채널 skip
    output_path = os.path.join(OUTPUT_DIR, f"{channel_name}.parquet")
    if os.path.exists(output_path):
        skipped_channels += 1
        tqdm.write(f"⏭️ {skipped_channels}  {channel_name} — 이미 처리됨, skip")
        continue

    parquets = sorted(glob.glob(os.path.join(glob.escape(channel_entry.path), "*.parquet")))
    if not parquets:
        tqdm.write(f"⏭️  {channel_name} — parquet 없음, skip")
        continue
    
    # ── Phase 1: 데이터 준비 ──
    tasks = []
    skip_no_text = 0
    skip_no_image = 0

    for pq in parquets:
        pq_file = os.path.basename(pq)
        video_name = pq_file[:-8]  # .parquet 제거

        df = pd.read_parquet(pq, columns=["frame_num", "ocr_texts", "ocr_scores", "ocr_xywha"])
        has_valid = df[df.apply(
            lambda r: any(s >= OCR_SCORE_THRESHOLD for s in json.loads(r["ocr_scores"])) if json.loads(r["ocr_texts"]) else False,
            axis=1
        )]
        if has_valid.empty:
            skip_no_text += 1
            continue

        n_sample = max(1, round(len(has_valid) * 0.01))
        sampled = has_valid.sample(n=n_sample, random_state=42)

        for _, row in sampled.iterrows():
            frame_num = row["frame_num"]
            image_path = os.path.join(FRAME_DIR, channel_name, video_name, f"frame_{frame_num:08d}.jpg")
            if not os.path.exists(image_path):
                skip_no_image += 1
                continue

            ocr_texts = json.loads(row["ocr_texts"])
            ocr_scores = json.loads(row["ocr_scores"])
            ocr_xywha = json.loads(row["ocr_xywha"])

            # score 필터링 (has_valid에서 최소 1개 보장됨)
            filtered = [(t, s, b) for t, s, b in zip(ocr_texts, ocr_scores, ocr_xywha) if s >= OCR_SCORE_THRESHOLD]
            ocr_texts, ocr_scores, ocr_xywha = zip(*filtered)
            ocr_texts, ocr_scores, ocr_xywha = list(ocr_texts), list(ocr_scores), list(ocr_xywha)

            regions = []
            for i, (text, bbox) in enumerate(zip(ocr_texts, ocr_xywha)):
                cx, cy, w, h, angle = bbox
                regions.append({
                    "id": i, "text": text,
                    "cx": round(cx), "cy": round(cy),
                    "w": round(w), "h": round(h),
                    "angle": round(angle, 1)
                })

            user_text = f"""OCR-detected text regions in this frame:
{json.dumps(regions, ensure_ascii=False)}

Classify each region. Respond ONLY with a JSON array:
[{{"id": 0, "classification": "telop"}}, {{"id": 1, "classification": "scene_text"}}, ...]"""

            with open(image_path, "rb") as f:
                img_b64 = base64.b64encode(f.read()).decode("utf-8")

            tasks.append({
                "video_name": video_name,
                "frame_num": int(frame_num),
                "ocr_texts": ocr_texts,
                "ocr_xywha": ocr_xywha,
                "img_b64": img_b64,
                "user_text": user_text,
            })

    # ── Phase 2: 동시 VLM 호출 ──
    channel_rows = []
    ch_errors = 0

    if tasks:
        pbar = tqdm(total=len(tasks), desc=f"📁 {channel_name}", leave=True)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(call_vlm, t): t for t in tasks}
            for future in as_completed(futures):
                result = future.result()
                task = result["task"]
                video_name = task["video_name"]
                ocr_texts = task["ocr_texts"]
                ocr_xywha = task["ocr_xywha"]

                times.append(result["elapsed"])

                if result["success"]:
                    is_telop = result["is_telop"]
                    for v in result["cls_map"].values():
                        if v == "telop":
                            total_telop += 1
                        elif v == "scene_text":
                            total_scene += 1
                        else:
                            total_unknown += 1
                    total_regions += len(ocr_texts)
                else:
                    errors += 1
                    ch_errors += 1
                    total_regions += len(ocr_texts)
                    is_telop = [None] * len(ocr_texts)
                    tqdm.write(f"  ⚠️ timeout/에러: {video_name} ({result['elapsed']:.1f}초) — {result['error']}")

                total_frames += 1
                channel_rows.append({
                    "video_name": video_name,
                    "frame_num": task["frame_num"],
                    "ocr_xywha": json.dumps(ocr_xywha),
                    "ocr_texts": json.dumps(ocr_texts, ensure_ascii=False),
                    "is_telop": json.dumps(is_telop),
                })

                pbar.set_postfix_str(f"ok={len(channel_rows)-ch_errors} err={ch_errors}", refresh=True)
                pbar.update(1)
        pbar.close()

    # 채널 처리 완료 → 저장
    n_skipped = skip_no_text + skip_no_image
    skip_detail = f" (skip {n_skipped}: 유효텍스트없음={skip_no_text}/이미지없음={skip_no_image})" if n_skipped else ""
    if channel_rows:
        pd.DataFrame(channel_rows).to_parquet(output_path, index=False)
        tqdm.write(f"  ✅ {channel_name} 저장 — {len(channel_rows)}프레임/{len(parquets)}개 영상, err={ch_errors}{skip_detail}")
    else:
        tqdm.write(f"  ⚠️ {channel_name} — 유효 프레임 없음{skip_detail}")

# 최종 요약
avg_time = sum(times) / len(times) if times else 0
tqdm.write("\n" + "=" * 60)
tqdm.write(f"📊 전체 처리 결과")
tqdm.write("=" * 60)
tqdm.write(f"  채널: {total_channels}개 (skip {skipped_channels} / 처리 {total_channels - skipped_channels})")
tqdm.write(f"  프레임: {total_frames}개")
tqdm.write(f"  총 소요시간: {sum(times):.1f}초 ({sum(times)/60:.1f}분)")
tqdm.write(f"  프레임당 평균: {avg_time:.2f}초")
tqdm.write(f"  총 텍스트 영역: {total_regions}개")
tqdm.write(f"  🟢 텔롭: {total_telop}개 ({total_telop/max(total_regions,1)*100:.1f}%)")
tqdm.write(f"  🔴 비텔롭: {total_scene}개 ({total_scene/max(total_regions,1)*100:.1f}%)")
tqdm.write(f"  ❓ unknown/에러: {total_unknown + errors}개")
tqdm.write(f"  에러: {errors}건")
tqdm.write("=" * 60)


⏭️ 1 고기남자 — 이미 처리됨, skip
⏭️ 2  &TEAM — 이미 처리됨, skip
⏭️ 3  14F 일사에프 — 이미 처리됨, skip
⏭️ 4  1MILLION Dance Studio — 이미 처리됨, skip
⏭️ 5  1theK Originals - 원더케이 오리지널 — 이미 처리됨, skip
⏭️ 6  1분미만 — 이미 처리됨, skip
⏭️ 7  1분요리 뚝딱이형 — 이미 처리됨, skip
⏭️ 8  2PM — 이미 처리됨, skip
⏭️ 9  5 분 Tricks — 이미 처리됨, skip
⏭️ 10  815머니톡 — 이미 처리됨, skip
⏭️ 11  83부부 — 이미 처리됨, skip
⏭️ 12  A2O Channel — 이미 처리됨, skip
⏭️ 13  AFC Asian Cup — 이미 처리됨, skip
⏭️ 14  AKMU — 이미 처리됨, skip
⏭️ 15  ALL THE K-POP — 이미 처리됨, skip
⏭️ 16  ALLDAY PROJECT — 이미 처리됨, skip
⏭️ 17  ARIKITCHEN (아리키친) — 이미 처리됨, skip
⏭️ 18  ARIRANG K-POP — 이미 처리됨, skip
⏭️ 19  ARTBEAT — 이미 처리됨, skip
⏭️ 20  ASEAN United FC — 이미 처리됨, skip
⏭️ 21  ASTRO 아스트로 — 이미 처리됨, skip
⏭️ 22  ATEEZ — 이미 처리됨, skip
⏭️ 23  Again 가요톱10 _ KBS KPOP Classic — 이미 처리됨, skip
⏭️ 24  Allblanc TV — 이미 처리됨, skip
⏭️ 25  Apink (에이핑크) — 이미 처리됨, skip
⏭️ 26  B Man 삐맨 — 이미 처리됨, skip
⏭️ 27  BABYMONSTER — 이미 처리됨, skip
⏭️ 28  BANGTANTV — 이미 처리됨, skip
⏭️ 29  BEATPELLA HOUSE — 이미 처리됨, skip
⏭️ 30  BLACKPINK — 이미 처리

📁 Chuther츄더[문에스더]: 100%|██████████| 208/208 [00:41<00:00,  4.97it/s, ok=208 err=0]


  ✅ Chuther츄더[문에스더] 저장 — 208프레임/100개 영상, err=0
⏭️ 53  Clay Adventure 클레이 모험 — 이미 처리됨, skip
⏭️ 54  CookieRun_ Kingdom — 이미 처리됨, skip
⏭️ 55  Cooking tree 쿠킹트리 — 이미 처리됨, skip
⏭️ 56  Creedence Clearwater Revival — 이미 처리됨, skip
⏭️ 57  DARA TV — 이미 처리됨, skip
⏭️ 58  DAY6 — 이미 처리됨, skip
⏭️ 59  DECO_27 — 이미 처리됨, skip
⏭️ 60  DGG — 이미 처리됨, skip
⏭️ 61  DJ SODA OFFICIAL — 이미 처리됨, skip
⏭️ 62  DOM Studio — 이미 처리됨, skip
⏭️ 63  DRAMA Voyage — 이미 처리됨, skip
⏭️ 64  DaftTaengk — 이미 처리됨, skip
⏭️ 65  Daily Busking — 이미 처리됨, skip
⏭️ 66  Dan Music — 이미 처리됨, skip
⏭️ 67  DanalEntertainment — 이미 처리됨, skip
⏭️ 68  DinoCore TUBA — 이미 처리됨, skip
⏭️ 69  Dreamcatcher official — 이미 처리됨, skip
⏭️ 70  EBS 컬렉션 - 라이프스타일 — 이미 처리됨, skip
⏭️ 71  EBS 컬렉션 - 사이언스 — 이미 처리됨, skip
⏭️ 72  EBS 키즈 — 이미 처리됨, skip
⏭️ 73  EBSCulture (EBS 교양) — 이미 처리됨, skip
⏭️ 74  EBSDocumentary (EBS 다큐) — 이미 처리됨, skip
⏭️ 75  ENHYPEN — 이미 처리됨, skip
⏭️ 76  EPIK HIGH — 이미 처리됨, skip
⏭️ 77  ETTV 이티티비 — 이미 처리됨, skip
⏭️ 78  EXO — 이미 처리됨, skip
⏭️ 79  English Singsin

📁 Lime Tube[라임튜브]: 100%|██████████| 223/223 [00:41<00:00,  5.44it/s, ok=223 err=0]


  ✅ Lime Tube[라임튜브] 저장 — 223프레임/100개 영상, err=0
⏭️ 154  Louis Vuitton — 이미 처리됨, skip
⏭️ 155  M2 — 이미 처리됨, skip
⏭️ 156  MAMAMOO — 이미 처리됨, skip
⏭️ 157  MAPPA CHANNEL — 이미 처리됨, skip
⏭️ 158  MAXIM KOREA — 이미 처리됨, skip
⏭️ 159  MBC PD수첩 — 이미 처리됨, skip
⏭️ 160  MBC every1 — 이미 처리됨, skip
⏭️ 161  MBC 라디오 시사 — 이미 처리됨, skip
⏭️ 162  MBCNEWS — 이미 처리됨, skip
⏭️ 163  MBCdrama — 이미 처리됨, skip
⏭️ 164  MBCentertainment — 이미 처리됨, skip
⏭️ 165  MBCkpop — 이미 처리됨, skip
⏭️ 166  MBN Entertainment — 이미 처리됨, skip
⏭️ 167  MBN MUSIC — 이미 처리됨, skip
⏭️ 168  MBN News — 이미 처리됨, skip
⏭️ 169  MEOVV — 이미 처리됨, skip
⏭️ 170  MKTV 김미경TV — 이미 처리됨, skip
⏭️ 171  MMTG 문명특급 — 이미 처리됨, skip
⏭️ 172  MONSTA X — 이미 처리됨, skip
⏭️ 173  MOVE Dance Studio — 이미 처리됨, skip
⏭️ 174  MTN 머니투데이방송 — 이미 처리됨, skip
⏭️ 175  MUPLY 뮤플리 — 이미 처리됨, skip
⏭️ 176  MUSIC&NEW 뮤직앤뉴 — 이미 처리됨, skip
⏭️ 177  MariAndFriends — 이미 처리됨, skip
⏭️ 178  Marine Ch. 宝鐘マリン — 이미 처리됨, skip
⏭️ 179  Mera — 이미 처리됨, skip
⏭️ 180  Mnet K-POP — 이미 처리됨, skip
⏭️ 181  Mnet TV — 이미 처리됨, skip
⏭

...

  ✅ 생존연구소 [동심파괴] 저장 — 510프레임/100개 영상, err=0
⏭️ 447  샾잉 #ing — 이미 처리됨, skip
⏭️ 448  서울의소리 VoiceOfSeoul — 이미 처리됨, skip


📁 서은이야기[SeoeunStory]: 100%|██████████| 204/204 [00:31<00:00,  6.45it/s, ok=204 err=0]


  ✅ 서은이야기[SeoeunStory] 저장 — 204프레임/100개 영상, err=0
⏭️ 449  서은일상이야기 — 이미 처리됨, skip
⏭️ 450  선미 SUNMI — 이미 처리됨, skip
⏭️ 451  성시경 SUNG SI KYUNG — 이미 처리됨, skip
⏭️ 452  성제준 — 이미 처리됨, skip
⏭️ 453  세바시 강연 Sebasi Talk — 이미 처리됨, skip
⏭️ 454  세아쌤Seah Ssam — 이미 처리됨, skip
⏭️ 455  셀코TV _ 잘 먹고 잘 사는 방법 — 이미 처리됨, skip
⏭️ 456  셰프 안성재 Chef Sung Anh — 이미 처리됨, skip
⏭️ 457  소녀의행성 Girlsplanet — 이미 처리됨, skip
⏭️ 458  소맥거핀 — 이미 처리됨, skip
⏭️ 459  솔라시도 solarsido — 이미 처리됨, skip
⏭️ 460  수리노을SuriNoel — 이미 처리됨, skip
⏭️ 461  순자엄마 — 이미 처리됨, skip
⏭️ 462  슈기님 — 이미 처리됨, skip
⏭️ 463  슈앤트리 SHU AND TREE — 이미 처리됨, skip
⏭️ 464  슛포러브 — 이미 처리됨, skip
⏭️ 465  스브스 예능맛집 — 이미 처리됨, skip
⏭️ 466  스브스뉴스 SUBUSUNEWS — 이미 처리됨, skip
⏭️ 467  스터디언 — 이미 처리됨, skip
⏭️ 468  스튜디오 그냥 — 이미 처리됨, skip
⏭️ 469  스튜디오 수제 — 이미 처리됨, skip
⏭️ 470  스튜디오 와플 - STUDIO WAFFLE — 이미 처리됨, skip
⏭️ 471  승비니 Seungbini — 이미 처리됨, skip
⏭️ 472  승우아빠 — 이미 처리됨, skip
⏭️ 473  시사포커스TV — 이미 처리됨, skip
⏭️ 474  시즌비시즌 Season B Season — 이미 처리됨, skip
⏭️ 475  식탁일기 table diary — 이미 처리됨, sk

📁 이지금 [IU Official]: 100%|██████████| 238/238 [00:34<00:00,  6.81it/s, ok=238 err=0]

  ✅ 이지금 [IU Official] 저장 — 238프레임/100개 영상, err=0
⏭️ 539  인생84 — 이미 처리됨, skip
⏭️ 540  임다TV — 이미 처리됨, skip
⏭️ 541  임영웅 — 이미 처리됨, skip
⏭️ 542  입질의추억TV jiminTV — 이미 처리됨, skip
⏭️ 543  입짧은햇님 — 이미 처리됨, skip
⏭️ 544  자이언트 펭TV — 이미 처리됨, skip
⏭️ 545  자취요리신 simple cooking — 이미 처리됨, skip
⏭️ 546  장지수 — 이미 처리됨, skip
⏭️ 547  재훍 영상툰 — 이미 처리됨, skip
⏭️ 548  전인구경제연구소 — 이미 처리됨, skip
⏭️ 549  정브르 — 이미 처리됨, skip
⏭️ 550  정선근 TV — 이미 처리됨, skip
⏭️ 551  정선호(Seonho Jung) — 이미 처리됨, skip
⏭️ 552  정세연의 라이프연구소 — 이미 처리됨, skip
⏭️ 553  정찬성 Korean Zombie — 이미 처리됨, skip
⏭️ 554  제이제이살롱드핏 — 이미 처리됨, skip
⏭️ 555  제이키아웃 JAYKEEOUT — 이미 처리됨, skip
⏭️ 556  조나단 — 이미 처리됨, skip
⏭️ 557  조선일보 — 이미 처리됨, skip
⏭️ 558  조재원 — 이미 처리됨, skip
⏭️ 559  주니토니 동요동화 - 키즈캐슬 — 이미 처리됨, skip
⏭️ 560  주부나라 — 이미 처리됨, skip
⏭️ 561  준우 — 이미 처리됨, skip
⏭️ 562  지무비 _ G Movie — 이미 처리됨, skip
⏭️ 563  지켜츄 Chuu Can Do It — 이미 처리됨, skip
⏭️ 564  직업의모든것 All about jobs — 이미 처리됨, skip
⏭️ 565  진영민yeongmin — 이미 처리됨, skip
⏭️ 566  진용진 — 이미 처리됨, skip
⏭️ 567  집나간아들 Runaway Son — 이

In [7]:
# %% 셀 6: 에러(null) 프레임 재처리
import os
import json
import base64
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
import httpx

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(120.0, connect=5.0)  # 타임아웃 늘림
)

MODEL_NAME = "Qwen/Qwen3.5-9B"
FRAME_DIR = "./data/2_frame_files"
OUTPUT_DIR = "./data/4_is_telop_results"
TIMEOUT = 120  # 타임아웃 늘림
MAX_WORKERS = 50  # 동시요청 줄임

SYSTEM_PROMPT = """You are a video frame analyzer that classifies OCR-detected text regions as either "telop" or "scene_text".

Telop: Text overlaid during post-production. Subtitles, captions, name plates, speech bubbles, decorative titles, reaction text, timestamps, channel logos, watermarks, and any graphical text composited onto the video frame. May be static or animated.

Scene text: Text physically existing in the 3D scene. Signage, clothing logos, book covers, product labels, screens within the scene, license plates, posters, handwritten notes, etc.

Key visual cues for telop:
- No perspective distortion (faces viewer regardless of camera angle)
- Consistent lighting unaffected by scene illumination
- Often has outline, drop shadow, stroke, or semi-transparent background
- Clean uniform font rendering
- May overlap scene objects without being occluded
- Designed for readability over video content

Key visual cues for scene text:
- Perspective distortion matching its surface
- Affected by scene lighting, shadows, reflections
- Surface texture visible beneath text
- May be partially occluded by other objects"""

OCR_SCORE_THRESHOLD = 0.4


def call_vlm(task):
    video_name = task["video_name"]
    img_b64 = task["img_b64"]
    user_text = task["user_text"]
    ocr_texts = task["ocr_texts"]

    t0 = time.time()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
                        {"type": "text", "text": user_text}
                    ]
                }
            ],
            max_tokens=4096,
            temperature=0.1,
            timeout=TIMEOUT,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        elapsed = time.time() - t0
        raw = response.choices[0].message.content.strip()
        cleaned = raw.removeprefix("```json").removesuffix("```").strip()
        classifications = json.loads(cleaned)
        cls_map = {item["id"]: item.get("classification", "unknown") for item in classifications}
        is_telop = [cls_map.get(i, "unknown") == "telop" for i in range(len(ocr_texts))]
        return {"success": True, "elapsed": elapsed, "is_telop": is_telop, "task": task}
    except Exception as e:
        elapsed = time.time() - t0
        return {"success": False, "elapsed": elapsed, "error": str(e)[:80], "task": task}


total_retried = 0
total_fixed = 0
total_still_error = 0

for pq_path in sorted(os.listdir(OUTPUT_DIR)):
    if not pq_path.endswith(".parquet"):
        continue
    channel_name = pq_path[:-8]  # .parquet 제거
    full_path = os.path.join(OUTPUT_DIR, pq_path)

    df = pd.read_parquet(full_path)
    # null 포함된 행 찾기
    error_mask = df["is_telop"].apply(lambda x: None in json.loads(x))
    if not error_mask.any():
        continue

    error_df = df[error_mask]
    tqdm.write(f"🔄 {channel_name} — 에러 {len(error_df)}건 재처리")

    # task 준비
    tasks = []
    for idx, row in error_df.iterrows():
        video_name = row["video_name"]
        frame_num = row["frame_num"]
        ocr_texts = json.loads(row["ocr_texts"])
        ocr_xywha = json.loads(row["ocr_xywha"])

        image_path = os.path.join(FRAME_DIR, channel_name, video_name, f"frame_{frame_num:08d}.jpg")
        if not os.path.exists(image_path):
            continue

        regions = []
        for i, (text, bbox) in enumerate(zip(ocr_texts, ocr_xywha)):
            cx, cy, w, h, angle = bbox
            regions.append({
                "id": i, "text": text,
                "cx": round(cx), "cy": round(cy),
                "w": round(w), "h": round(h),
                "angle": round(angle, 1)
            })

        user_text = f"""OCR-detected text regions in this frame:
{json.dumps(regions, ensure_ascii=False)}

Classify each region. Respond ONLY with a JSON array:
[{{"id": 0, "classification": "telop"}}, {{"id": 1, "classification": "scene_text"}}, ...]"""

        with open(image_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode("utf-8")

        tasks.append({
            "df_idx": idx,
            "video_name": video_name,
            "frame_num": int(frame_num),
            "ocr_texts": ocr_texts,
            "ocr_xywha": ocr_xywha,
            "img_b64": img_b64,
            "user_text": user_text,
        })

    if not tasks:
        continue

    # VLM 호출
    fixed = 0
    still_error = 0
    pbar = tqdm(total=len(tasks), desc=f"🔄 {channel_name}", leave=True)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(call_vlm, t): t for t in tasks}
        for future in as_completed(futures):
            result = future.result()
            task = result["task"]
            idx = task["df_idx"]

            if result["success"]:
                df.at[idx, "is_telop"] = json.dumps(result["is_telop"])
                fixed += 1
            else:
                still_error += 1
                tqdm.write(f"  ⚠️ 재실패: {task['video_name']} — {result['error']}")

            pbar.update(1)
    pbar.close()

    # 덮어쓰기 저장
    df.to_parquet(full_path, index=False)
    tqdm.write(f"  ✅ {channel_name} — 수정 {fixed}건, 여전히 에러 {still_error}건")

    total_retried += len(tasks)
    total_fixed += fixed
    total_still_error += still_error

tqdm.write(f"\n📊 재처리 완료: 시도 {total_retried}, 수정 {total_fixed}, 실패 {total_still_error}")

🔄 오빠두엑셀 l 엑셀 강의 대표채널 — 에러 11건 재처리


🔄 오빠두엑셀 l 엑셀 강의 대표채널:   0%|          | 0/11 [00:00<?, ?it/s]             

  ⚠️ 재실패: 엑셀 고수만 아는, '이동 옵션'의 숨겨진 기능 (필드 정리 Tip⚡) #shorts__qAYI0aBS-vU — Error code: 400 - {'object': 'error', 'message': "Requested token count exceeds 


🔄 오빠두엑셀 l 엑셀 강의 대표채널:  36%|███▋      | 4/11 [00:00<00:00, 16.44it/s]             

  ⚠️ 재실패: 엑셀 고수만 아는, '이동 옵션'의 숨겨진 기능 (필드 정리 Tip⚡) #shorts__qAYI0aBS-vU — Error code: 400 - {'object': 'error', 'message': "Requested token count exceeds 
  ⚠️ 재실패: 엑셀 고수만 아는, '이동 옵션'의 숨겨진 기능 (필드 정리 Tip⚡) #shorts__qAYI0aBS-vU — Error code: 400 - {'object': 'error', 'message': "Requested token count exceeds 
  ⚠️ 재실패: 엑셀 고수만 아는, '이동 옵션'의 숨겨진 기능 (필드 정리 Tip⚡) #shorts__qAYI0aBS-vU — Error code: 400 - {'object': 'error', 'message': "Requested token count exceeds 
  ⚠️ 재실패: 엑셀 그룹별 데이터 나누기, 단축키 5초 해결법⚡️(알아두면 정말 편해요!) #shorts__ZEOmu1joW3I — Error code: 400 - {'object': 'error', 'message': "Requested token count exceeds 


...

  ⚠️ 재실패: 엑셀 여러개 시트 동시 비교, 알아두면 정말 편리합니다 (5초 해결방법⚡)__JicrMrhsXBI — Error code: 400 - {'object': 'error', 'message': "The input (25940 tokens) is lo
  ⚠️ 재실패: 엑셀 여러개 시트 동시 비교, 알아두면 정말 편리합니다 (5초 해결방법⚡)__JicrMrhsXBI — Error code: 400 - {'object': 'error', 'message': "The input (16556 tokens) is lo
  ⚠️ 재실패: 엑셀 여러개 시트 동시 비교, 알아두면 정말 편리합니다 (5초 해결방법⚡)__JicrMrhsXBI — Error code: 400 - {'object': 'error', 'message': "The input (22022 tokens) is lo


🔄 오빠두엑셀 l 엑셀 강의 대표채널:  82%|████████▏ | 9/11 [00:00<00:00, 20.15it/s]             

  ⚠️ 재실패: 엑셀 인쇄 문제, 더 이상 고민하지 마세요! ｜ 세로 긴 문서 인쇄하는법⚡ #shorts__1qUy1LGDpv4 — Error code: 400 - {'object': 'error', 'message': "The input (16953 tokens) is lo


🔄 오빠두엑셀 l 엑셀 강의 대표채널: 100%|██████████| 11/11 [00:00<00:00, 19.27it/s]            


  ⚠️ 재실패: 엑셀로 인터넷 자료를 1초 안에 불러오는 방법⚡(정말 편리합니다!)__rrde0n6dCkw — Error code: 400 - {'object': 'error', 'message': "The input (23579 tokens) is lo
  ✅ 오빠두엑셀 l 엑셀 강의 대표채널 — 수정 0건, 여전히 에러 11건

📊 재처리 완료: 시도 11, 수정 0, 실패 11
